# W5 Day3 (05/03 周六): GraphRAG 原理 ★

## 学习目标
- **理论 (2h)**: Microsoft GraphRAG; LightRAG (图索引+双层检索+增量); 本体如何融入
- **实践 (3h)**: 精读 LightRAG 源码 + 分析本体约束抽取切入点
- **产出物**: LightRAG 源码笔记

## 核心问题 (面试高频)
1. GraphRAG 和 Naive RAG 的本质区别？什么场景下 GraphRAG 更好？
2. Microsoft GraphRAG 的索引构建流程？为什么成本高？
3. LightRAG 怎么降低成本？双层检索是什么？
4. 本体 (Ontology) 怎么融入 GraphRAG？
5. GraphRAG 的增量更新怎么实现？
6. GraphRAG 的评估方法？

---

## 目录
1. [Naive RAG 的局限](#1)
2. [Microsoft GraphRAG](#2)
3. [LightRAG 架构](#3)
4. [本体增强 GraphRAG](#4)
5. [源码分析: LightRAG 核心模块](#5)
6. [GraphRAG 评估](#6)
7. [总结 & 思考题](#7)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

TERRACOTTA = '#c2553a'
SAGE = '#5a6b4a'
INK = '#2d2a26'

print("环境就绪")
print("=" * 60)

---
## 1. Naive RAG 的局限 <a id='1'></a>

### 1.1 Naive RAG 的三大问题

| 问题 | 描述 | 例子 |
|---|---|---|
| **全局问题** | 无法回答需要跨文档综合的问题 | "这个项目的主要风险是什么？" |
| **多跳推理** | 需要多步推理才能回答 | "A公司的CEO毕业于哪所大学？" |
| **关系理解** | 无法捕捉实体间的关系 | "哪些供应商和同一个客户合作？" |

### 1.2 为什么需要图结构？

```
Naive RAG: 文档 → 切分 → 向量检索 (丢失了 chunk 之间的关系)

GraphRAG: 文档 → 实体/关系抽取 → 知识图谱 → 图检索 (保留了结构信息)
```

### 1.3 GraphRAG 的核心思想

**在索引阶段构建知识图谱，在检索阶段利用图结构进行推理。**

---
## 2. Microsoft GraphRAG <a id='2'></a>

### 2.1 索引构建流程

```
Step 1: 实体抽取
  文档 → LLM → (实体, 关系, 描述) 三元组
  "墙体A长3.5m" → (墙体A, 长度, 3.5m)

Step 2: 社区检测
  知识图谱 → Leiden 算法 → 社区层次结构
  (类似公司组织架构: 部门→团队→个人)

Step 3: 社区摘要
  每个社区 → LLM → 生成摘要
  (用社区内的实体和关系作为上下文)

Step 4: 向量化
  实体描述 + 社区摘要 → Embedding → 向量索引
```

### 2.2 两种检索模式

| 模式 | 适用问题 | 检索方式 |
|---|---|---|
| **Local Search** | 具体实体相关 | 实体→邻居→相关社区摘要 |
| **Global Search** | 全局性问题 | 社区摘要→Map-Reduce 汇总 |

### 2.3 成本问题

| 环节 | 成本 |
|---|---|
| 实体抽取 | **高** (每段文档都要调 LLM) |
| 社区摘要 | 中 (每个社区调一次 LLM) |
| 查询 | 低 (只在图上搜索) |

→ 索引阶段的 LLM 调用成本是主要瓶颈

In [ ]:
# ============ Microsoft GraphRAG 流程模拟 ============

class SimulatedGraphRAG:
    """模拟 Microsoft GraphRAG 的核心流程"""
    
    def __init__(self):
        self.entities = {}  # name → {type, description}
        self.relations = []  # (src, dst, type, description)
        self.communities = {}  # id → {entities, summary}
        self.llm_calls = 0
    
    def extract_entities(self, text: str) -> list:
        """模拟 LLM 实体抽取"""
        self.llm_calls += 1
        # 模拟抽取结果
        entities = [
            {'name': '墙体A', 'type': 'Wall', 'desc': '长3.5m的承重墙'},
            {'name': '门D1', 'type': 'Door', 'desc': '宽0.6m的疏散门'},
            {'name': '窗W1', 'type': 'Window', 'desc': '宽1.5m的外窗'},
            {'name': '柱C1', 'type': 'Column', 'desc': '350x350mm混凝土柱'},
        ]
        relations = [
            ('墙体A', '门D1', 'hasDoor', '墙体A上安装了门D1'),
            ('墙体A', '窗W1', 'hasWindow', '墙体A上安装了窗W1'),
            ('柱C1', '墙体A', 'supports', '柱C1支撑墙体A'),
        ]
        return entities, relations
    
    def build_communities(self):
        """模拟 Leiden 社区检测"""
        self.communities = {
            0: {'entities': ['墙体A', '门D1', '窗W1'], 'summary': '外墙系统: 墙体A包含门D1和窗W1'},
            1: {'entities': ['柱C1', '墙体A'], 'summary': '承重系统: 柱C1支撑墙体A'},
        }
        self.llm_calls += len(self.communities)
    
    def local_search(self, query: str) -> str:
        """Local Search: 实体→邻居→社区摘要"""
        self.llm_calls += 1
        return "Local search: 找到实体'墙体A'及其邻居, 社区摘要: 外墙系统..."
    
    def global_search(self, query: str) -> str:
        """Global Search: Map-Reduce 社区摘要"""
        self.llm_calls += len(self.communities) + 1  # map + reduce
        return "Global search: 综合所有社区摘要..."


grag = SimulatedGraphRAG()

# 索引
entities, relations = grag.extract_entities(sample_doc if 'sample_doc' in dir() else "建筑安全审查")
grag.build_communities()

print("Microsoft GraphRAG 索引结果:")
print(f"  实体: {len(entities)} 个")
print(f"  关系: {len(relations)} 条")
print(f"  社区: {len(grag.communities)} 个")
print(f"  LLM 调用: {grag.llm_calls} 次")

# 成本估算
print(f"\n成本估算 (假设文档 100 页):")
print(f"  实体抽取: ~100 次 LLM 调用 (每页1次)")
print(f"  社区摘要: ~20 次 LLM 调用")
print(f"  总计: ~120 次 LLM 调用 ≈ $1-3 (GPT-4)")
print(f"  ⚠️ 大文档 (1000页): ~$10-30 → 成本是主要问题")

---
## 3. LightRAG 架构 <a id='3'></a>

### 3.1 核心改进

| 维度 | Microsoft GraphRAG | LightRAG |
|---|---|---|
| **实体抽取** | LLM (高成本) | LLM + 优化 prompt |
| **社区检测** | Leiden + LLM 摘要 | **无社区** (直接用图) |
| **检索** | Local/Global | **双层检索** (低层+高层) |
| **增量更新** | 不支持 | **支持** |
| **成本** | 高 | 中 |

### 3.2 双层检索

```
用户 Query
    ↓
高层检索: Query → 关键词 → 知识图谱中的相关实体/关系
    ↓
低层检索: 相关实体 → 原始文档 chunks (保留原始上下文)
    ↓
综合生成
```

### 3.3 增量更新

```
新文档 → 实体抽取 → 与已有图合并 (去重/更新)
                   → 更新受影响的 chunks
```

- 不需要重建整个图
- 只更新变化的部分

In [ ]:
# ============ LightRAG 双层检索模拟 ============

class LightRAGSimulator:
    """模拟 LightRAG 的双层检索"""
    
    def __init__(self):
        self.kg = {}  # entity → {relations, chunks}
        self.chunks = {}  # chunk_id → text
    
    def index(self, docs):
        """模拟索引: 文档→实体+关系+chunks"""
        for i, doc in enumerate(docs):
            chunk_id = f'chunk_{i}'
            self.chunks[chunk_id] = doc
            # 模拟实体抽取
            entities = self._extract(doc)
            for ent in entities:
                if ent not in self.kg:
                    self.kg[ent] = {'relations': [], 'chunks': []}
                self.kg[ent]['chunks'].append(chunk_id)
    
    def _extract(self, text):
        """简易实体抽取"""
        keywords = ['墙', '门', '窗', '柱', '楼梯', '消防', '安全', '审查', '规范']
        return [k for k in keywords if k in text]
    
    def retrieve(self, query):
        """双层检索"""
        # 高层: 关键词匹配实体
        query_entities = self._extract(query)
        print(f"  高层检索: 匹配实体 {query_entities}")
        
        # 低层: 通过实体找到相关 chunks
        relevant_chunks = set()
        for ent in query_entities:
            if ent in self.kg:
                relevant_chunks.update(self.kg[ent]['chunks'])
        
        print(f"  低层检索: 找到 {len(relevant_chunks)} 个相关 chunks")
        return [self.chunks[cid] for cid in relevant_chunks]


# 模拟
lrag = LightRAGSimulator()
docs = [
    "承重墙的最小厚度应为200mm。",
    "疏散通道的门净宽度不应小于0.9m。",
    "消防车道的宽度不应小于4m。",
    "疏散楼梯的净宽度不应小于1.1m。",
    "防火墙上窗户间距不应小于1.0m。",
]
lrag.index(docs)

print(f"知识图谱: {len(lrag.kg)} 个实体")
for ent, info in lrag.kg.items():
    print(f"  {ent}: {len(info['chunks'])} chunks")

print(f"\n查询: '墙体和门的安全要求'")
results = lrag.retrieve('墙体和门的安全要求')
for i, r in enumerate(results, 1):
    print(f"  [{i}] {r}")

---
## 4. 本体增强 GraphRAG <a id='4'></a>

### 你的博士工作与 GraphRAG 的结合

```
传统 GraphRAG:
  文档 → LLM 自由抽取 → 实体/关系 (可能不准确、不一致)

本体增强 GraphRAG:
  文档 → 本体约束抽取 → 实体/关系 (结构化、一致、可推理)
```

### 本体的三大作用

| 作用 | 描述 | 例子 |
|---|---|---|
| **约束抽取** | 限定实体/关系的类型 | 只抽取 Wall/Door/Window 类型的实体 |
| **指导检索** | 本体层次帮助定位 | "安全审查"→"墙体"→"承重墙" |
| **推理增强** | 利用本体公理推理 | Wall ⊓ Door = ⊥ → 墙不是门 |

### 抽取对比

| | 自由抽取 | 本体约束抽取 |
|---|---|---|
| **实体类型** | 任意 (LLM 自由发挥) | 预定义 (Wall/Door/...) |
| **关系类型** | 任意 | 预定义 (hasPart/connectsTo/...) |
| **一致性** | 低 (同义词/歧义) | 高 (统一本体) |
| **可推理** | 不可 | 可 (描述逻辑推理) |
| **成本** | 低 (一次 LLM 调用) | 中 (需要本体+约束检查) |

In [ ]:
# ============ 本体约束抽取 vs 自由抽取 对比 ============

class OntologyConstrainedExtractor:
    """本体约束的实体抽取器"""
    
    def __init__(self, ontology):
        self.ontology = ontology
    
    def extract(self, text):
        """只抽取符合本体定义的实体和关系"""
        entities = []
        relations = []
        
        # 实体抽取: 只匹配本体中的类
        for cls in self.ontology['classes']:
            if cls.lower() in text.lower() or cls in text:
                entities.append({'name': cls, 'type': cls})
        
        # 关系抽取: 只匹配本体中的属性
        for prop in self.ontology['properties']:
            if prop in text:
                # 简单匹配: 找到属性附近的实体
                for ent in entities:
                    relations.append({
                        'source': ent['name'],
                        'type': prop,
                        'evidence': text[:50]
                    })
        
        return entities, relations


# 本体定义
building_ontology = {
    'classes': ['Wall', 'Door', 'Window', 'Column', 'Beam', 'Stair', 'Floor', 'Room'],
    'properties': ['hasLength', 'hasWidth', 'hasArea', 'connectsTo', 'locatedIn', 'supports'],
    'constraints': [
        'Wall ⊓ Door = ⊥',  # 墙和门不相交
        'hasLength domain BuildingElement',
        'connectsTo domain BuildingElement',
    ]
}

extractor = OntologyConstrainedExtractor(building_ontology)

test_texts = [
    "承重墙Wall的长度hasLength为3.5m，位于房间Room_101中。",
    "门Door连接connectsTo到墙体Wall，宽度hasWidth为0.9m。",
    "柱Column支撑supports梁Beam，截面400x400mm。",
]

print("本体约束抽取结果:")
for text in test_texts:
    ents, rels = extractor.extract(text)
    print(f"\n  文本: {text[:40]}...")
    print(f"  实体: {[e['type'] for e in ents]}")
    print(f"  关系: {[(r['source'], r['type']) for r in rels]}")

print("\n💡 本体约束的优势:")
print("  1. 实体类型统一 (不会出现 '墙壁' vs '墙体' 的歧义)")
print("  2. 关系类型有意义 (不会抽取到无意义的关系)")
print("  3. 可以做一致性检查 (违反本体约束的抽取结果会被过滤)")

---
## 5. 源码分析: LightRAG 核心模块 <a id='5'></a>

### 5.1 LightRAG 代码结构

```
lightrag/
├── lightrag.py          # 主类 LightRAG
├── operate.py           # 核心操作 (抽取/索引/检索)
├── prompt.py            # Prompt 模板
├── utils.py             # 工具函数
└── storage/             # 存储后端
    ├── nano_vector.py   # 向量存储
    └── nano_graph.py    # 图存储
```

### 5.2 关键函数

| 函数 | 功能 | 调用频率 |
|---|---|---|
| `extract_entities` | LLM 抽取实体/关系 | 高 (索引时) |
| `build_graph` | 构建知识图谱 | 1次 (索引时) |
| `retrieve` | 双层检索 | 每次查询 |
| `generate_response` | LLM 生成答案 | 每次查询 |

### 5.3 实体抽取 Prompt (LightRAG)

```python
PROMPT = """
从以下文本中提取实体和关系。
实体类型: {entity_types}
关系类型: {relation_types}

输出 JSON 格式:
{"entities": [{"name": ..., "type": ..., "description": ...}],
 "relations": [{"source": ..., "target": ..., "type": ..., "description": ...}]}

文本: {text}
"""
```

### 5.4 本体融入切入点

可以在以下环节融入本体:

1. **抽取 Prompt**: 将本体的类/属性作为 `entity_types` / `relation_types`
2. **后处理**: 用本体约束过滤不合规的抽取结果
3. **检索**: 用本体层次扩展查询 (如 "安全" → "墙体安全" + "消防安全")
4. **推理**: 用本体公理做额外推理 (如继承关系传播)

---
## 6. GraphRAG 评估 <a id='6'></a>

### 评估维度

| 维度 | 指标 | Naive RAG | GraphRAG |
|---|---|---|---|
| **全局问题** | Global Recall | 差 | 好 |
| **多跳推理** | Multi-hop Accuracy | 中 | 好 |
| **具体事实** | Factual Precision | 好 | 好 |
| **成本** | LLM Calls | 低 | 高 |
| **延迟** | Query Latency | 低 | 中 |

### 什么时候用 GraphRAG？

- 文档之间有丰富的关系 (如知识图谱、法律条文、学术论文)
- 需要回答全局性/综合性问题
- 需要多跳推理
- 对一致性要求高 (如合规审查)

---
## 7. 总结 & 思考题 <a id='7'></a>

### 今日核心知识点

1. **GraphRAG**: 在索引阶段构建知识图谱，利用图结构进行推理
2. **Microsoft GraphRAG**: 实体抽取→社区检测→社区摘要，成本高
3. **LightRAG**: 双层检索+增量更新，降低成本
4. **本体增强**: 约束抽取+指导检索+推理增强
5. **适用场景**: 关系密集/全局问题/多跳推理/一致性要求高
6. **P2 项目**: 本体增强 GraphRAG 是你的核心项目

### 面试高频问题

1. **GraphRAG vs Naive RAG？** Naive RAG 丢失文档间关系; GraphRAG 保留图结构
2. **Microsoft GraphRAG 为什么贵？** 社区检测+摘要需要大量 LLM 调用
3. **LightRAG 怎么降低成本？** 去掉社区层，用双层检索替代
4. **本体怎么融入 GraphRAG？** 约束抽取+统一实体类型+推理增强
5. **GraphRAG 的增量更新？** 只更新变化的实体/关系，不重建全图

In [ ]:
print("=" * 60)
print("W5 Day3 完成!")
print("=" * 60)
print("""
今日成果:
  ✅ Naive RAG 局限性分析
  ✅ Microsoft GraphRAG 流程模拟 (实体抽取→社区→摘要)
  ✅ LightRAG 双层检索模拟
  ✅ 本体约束抽取 vs 自由抽取对比
  ✅ LightRAG 源码结构分析
  ✅ 本体融入 GraphRAG 的 4 个切入点

关键结论:
  • GraphRAG 适合关系密集/全局问题/多跳推理场景
  • LightRAG 通过去掉社区层降低成本
  • 本体增强 = 更准确的抽取 + 更一致的图 + 可推理
  • 这是 P2 项目 (本体增强 GraphRAG) 的理论基础
""")